In [2]:
from pynwb import NWBHDF5IO
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

In [3]:
file_path = r"C:\Users\kianb\Projects\mouse-v1-research\data\session_757216464\session_757216464.nwb"
io=NWBHDF5IO(file_path, mode="r")
nwbfile = io.read()
#print(nwbfile.session_id)
#print(nwbfile.session_description)
#print(nwbfile.session_start_time)
#print(nwbfile.institution)

#print("Total units:", len(nwbfile.units))

#print("Unit columns:", nwbfile.units.colnames)
units_df = nwbfile.units.to_dataframe()
units_df = units_df.reset_index(names="unit_id")
#print(units_df.head())


channels = pd.read_csv("../data/channels.csv")

In [4]:
#print(units_df['spike_times'])
merged_df = pd.merge(units_df, channels, left_on='peak_channel_id', right_on='id', how='left')
visp_units = merged_df[merged_df["ecephys_structure_acronym"] == "VISp"]

In [5]:
selected_units = visp_units[(visp_units['amplitude_cutoff'] < 0.1) & (visp_units['isi_violations'] < 0.5) & (visp_units['presence_ratio']>0.9) & (visp_units['quality'] == 'good')]
#print(Quality_pass['spike_times'],len(Quality_pass['spike_times']))

In [6]:
gratings = nwbfile.intervals['drifting_gratings_presentations']
gratings_df = gratings.to_dataframe()

valid_gratings = gratings_df[
    (gratings_df['orientation'].notna()) &
    (gratings_df['temporal_frequency'].notna())
]

print("Valid presentations:", len(valid_gratings))

Valid presentations: 600


In [7]:
#print(selected_units['spike_times'])
print(selected_units.columns.tolist())

['unit_id', 'waveform_halfwidth', 'waveform_duration', 'silhouette_score', 'nn_hit_rate', 'amplitude_cutoff', 'quality', 'spread', 'isi_violations', 'presence_ratio', 'peak_channel_id', 'nn_miss_rate', 'isolation_distance', 'PT_ratio', 'local_index_x', 'snr', 'velocity_below', 'd_prime', 'firing_rate', 'velocity_above', 'max_drift', 'cumulative_drift', 'repolarization_slope', 'amplitude', 'cluster_id', 'l_ratio', 'recovery_slope', 'spike_times', 'spike_amplitudes', 'waveform_mean', 'id', 'ecephys_probe_id', 'local_index_y', 'probe_horizontal_position', 'probe_vertical_position', 'anterior_posterior_ccf_coordinate', 'dorsal_ventral_ccf_coordinate', 'left_right_ccf_coordinate', 'ecephys_structure_id', 'ecephys_structure_acronym']


In [8]:
X = np.zeros((len(valid_gratings), len(selected_units)), dtype=int)
for i, presentation in enumerate(valid_gratings.itertuples()):
    start = presentation.start_time
    stop = presentation.stop_time
    for j, val in enumerate(selected_units.itertuples()):
        sTimes = val.spike_times
        count = np.sum((sTimes >= start) & (sTimes < stop))
        X[i,j] = count
    print(i, start, stop)

0 1586.1135845781948 1588.115244578195
1 1589.116094996702 1591.1177749967019
2 1592.1186045781951 1594.120274578195
3 1595.121124996702 1597.1227849967022
4 1598.123624578195 1600.1252945781948
5 1601.126144996702 1603.127794996702
6 1604.128654578195 1606.1303345781948
7 1607.1311549967022 1609.132834996702
8 1613.136174996702 1615.1378349967022
9 1616.138664578195 1618.140374578195
10 1619.141184996702 1621.142844996702
11 1622.143704578195 1624.145404578195
12 1625.1462149967024 1627.147884996702
13 1628.148734578195 1630.1503845781951
14 1631.151224996702 1633.1529049967019
15 1634.1537445781948 1636.1553945781948
16 1637.156254996702 1639.157924996702
17 1640.158744578195 1642.1604245781948
18 1643.1612549967024 1645.1629449967022
19 1646.163784578195 1648.1654745781948
20 1649.1662849967022 1651.167954996702
21 1652.1688045781948 1654.1704645781952
22 1655.171294996702 1657.172974996702
23 1658.173774578195 1660.1754745781948
24 1661.176324996702 1663.1779849967022
25 1664.17888

In [9]:
print(X.shape)
print(X[:3, :5])

(600, 87)
[[ 1  7 22 43  1]
 [19 50 33 21 28]
 [ 0 19 36 12  0]]


In [10]:
Y = valid_gratings['orientation'].to_numpy()
print(Y.shape)

(600,)


In [11]:
values, counts = np.unique(Y, return_counts=True)
result_2d = np.column_stack((values, counts))
print(result_2d)

[[  0.  75.]
 [ 45.  75.]
 [ 90.  75.]
 [135.  75.]
 [180.  75.]
 [225.  75.]
 [270.  75.]
 [315.  75.]]


In [12]:
presentation_ids = valid_gratings.index
units_ids = selected_units['unit_id']
print(presentation_ids.shape)
print(units_ids.shape)
print(presentation_ids.is_unique)
print(units_ids.is_unique)
print(Y[0] == valid_gratings['orientation'].iloc[0])
print(np.all(X >= 0))
print(np.array_equal(Y, valid_gratings['orientation'].to_numpy()))

(600,)
(87,)
True
True
True
True
True


In [13]:
spike_times = selected_units.iloc[20]['spike_times']

start = valid_gratings.iloc[100]['start_time']
stop = valid_gratings.iloc[100]['stop_time']
start=spike_times.searchsorted(start)
stop=spike_times.searchsorted(stop)
number=stop-start

In [14]:
print(X[100, 20] == number)

True


In [15]:
check_arr = [(0, 0), (1, 1), (100, 20), (300, 40), (599, 86)]

for i in range(len(check_arr)):
    spike_times_arr = selected_units.iloc[check_arr[i][1]]['spike_times']
    start = valid_gratings.iloc[check_arr[i][0]]['start_time']
    stop = valid_gratings.iloc[check_arr[i][0]]['stop_time']

    if np.all(np.diff(spike_times_arr) >= 0):
        start = spike_times_arr.searchsorted(start)
        stop = spike_times_arr.searchsorted(stop)
    else:
        print("Spike timestamps are not sorted!")
        break

    number = stop - start
    print(X[check_arr[i][0], check_arr[i][1]] == number)

True
True
True
True
True


In [19]:
temporal_frequencies = valid_gratings['temporal_frequency'].to_numpy()
np.save("../results/temporal_frequencies.npy", temporal_frequencies)

stimulus_block = valid_gratings['stimulus_block'].to_numpy()
np.save("../results/stimulus_block.npy", stimulus_block)

print(temporal_frequencies.shape)
print(stimulus_block.shape)

(600,)
(600,)


id
0      2.0
1      2.0
2      2.0
3      2.0
4      2.0
      ... 
625    7.0
626    7.0
627    7.0
628    7.0
629    7.0
Name: stimulus_block, Length: 600, dtype: float64